# 🔮 W9: 예지보전 & 이상감지 파이프라인 (v2)
**hexa-1 — Week 9** | ⏱️ 60분 | 🎯 3σ 이상감지 + 자동 알림 메시지

> 통계적 이상감지(3-Sigma Rule): 평균 ± 3σ를 벗어나면 이상 신호

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q pandas matplotlib scipy
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_nanum=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _nanum:
    fm.fontManager.addfont(_nanum[0])
    plt.rcParams['font.family']=fm.FontProperties(fname=_nanum[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
print('✅ 환경 설정 완료')


## Step 1: 데이터 로드 (이상치 포함 샘플)

In [ ]:
import requests, io
url='https://raw.githubusercontent.com/Reasonofmoon/hexa-1/main/scripts/oee_sample_data.csv'
try:
    df=pd.read_csv(url,encoding='utf-8-sig')
    print(f'✅ {len(df)}행 로드')
except:
    from google.colab import files; up=files.upload()
    if not up: raise RuntimeError('데이터 필요')
    df=pd.read_csv(io.BytesIO(list(up.values())[0]),encoding='utf-8-sig')

date_col=next((c for c in df.columns if '날짜' in c),df.columns[0])
defect_col=next((c for c in df.columns if '불량' in c and '수' in c),None)
total_col=next((c for c in df.columns if '생산량' in c),None)
df[date_col]=pd.to_datetime(df[date_col])
df['불량률']=df[defect_col]/df[total_col]*100

# ✅ 이상치 보장: 학습용으로 의도적으로 2건 주입
df_viz=df.copy()
outlier_idx=[5,18]  # 6번째, 19번째 행에 이상치 삽입
for idx in outlier_idx:
    if idx < len(df_viz):
        df_viz.loc[df_viz.index[idx],'불량률']=df_viz['불량률'].mean()+4*df_viz['불량률'].std()
print(f'✅ 데이터 준비 완료 (이상치 {len(outlier_idx)}건 삽입)')
df_viz.head(3)


## Step 2: 3σ 이상감지

In [ ]:
mean=df_viz['불량률'].mean()
std=df_viz['불량률'].std()
upper=mean+3*std
lower=max(0,mean-3*std)
df_viz['이상']=(df_viz['불량률']>upper)|(df_viz['불량률']<lower)
anomalies=df_viz[df_viz['이상']]
print(f'📊 정상 범위: {lower:.2f}% ~ {upper:.2f}%')
print(f'⚠️ 이상 감지: {len(anomalies)}건 / 전체 {len(df_viz)}건')
print(anomalies[[date_col,'불량률']].to_string(index=False))


## Step 3: 시각화 + 알림 메시지

In [ ]:
fig,ax=plt.subplots(figsize=(14,5))
ax.plot(df_viz[date_col],df_viz['불량률'],'b-o',label='불량률',markersize=4)
ax.axhline(upper,color='red',linestyle='--',label=f'상한({upper:.2f}%)')
ax.axhline(mean,color='green',linestyle='-',label=f'평균({mean:.2f}%)')
ax.scatter(anomalies[date_col],anomalies['불량률'],
           color='red',s=120,zorder=5,label=f'이상({len(anomalies)}건)')
ax.set_title('예지보전: 불량률 이상감지 (3σ 규칙)')
ax.set_ylabel('불량률 (%)')
ax.legend()
plt.tight_layout()
plt.savefig('anomaly_detection.png',dpi=150,bbox_inches='tight')
plt.show()
print('\n🚨 [예지보전 알림 메시지]')
for _,row in anomalies.iterrows():
    print(f' - {row[date_col].date()}: 불량률 {row["불량률"]:.2f}% 이상 감지!')
print('→ W7 Slack 노트북과 연결해서 자동 알림 발송 가능!')


## Step 4: 다운로드

In [ ]:
df_viz.to_csv('anomaly_results.csv',index=False,encoding='utf-8-sig')
from google.colab import files
files.download('anomaly_results.csv')
files.download('anomaly_detection.png')
print('✅ 완료!')


---
## 🔥 확장 과제
1. σ 배수를 2로 줄여 더 민감한 감지 → `upper=mean+2*std`
2. W07 Slack과 연결: 이상 시 자동 채널 알림
3. 이동평균(rolling mean) 기반 이상감지로 업그레이드

In [ ]:
# === Gemini AI 분석 (자동 추가됨) ===
!pip install -q google-generativeai
try:
    from google.colab import userdata; _api=userdata.get('GEMINI_API_KEY')
except: _api=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=_api)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
_model=genai.GenerativeModel('gemini-2.5-flash'); print('✅ Gemini 분석 준비')


## 🤖 AI 인사이트 분석 (Gemini)

In [ ]:
_p=f"""제조업 AI 컨설턴트로서 예측정비 이상감지 결과를 분석하세요.
핵심 인사이트 3가지 + 즉시 개선 액션. 마크다운."""
_resp=_model.generate_content(_p)
print(_resp.text)
